# MACE from first principles

A Pass 3 derivation-and-implementation notebook.

**Target reader:** a machine-learning practitioner who understands tensors, linear layers, gradients, and ordinary message passing, but may know nothing about geometric deep learning.

**Authorship rule:** write every explanation and implementation yourself. This notebook supplies the sequence, questions, shape contracts, visual exercises, and mathematical checks.

## 0. Setup — imports

In [2]:
import torch
import math
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.graph_objects as go
from e3nn import o3

## Learning route

For every topic: write plain English → connect it to familiar ML → record shapes → code a tiny example → graph it → test the mathematical identity → draft the blog paragraph.

- [ ] atomic geometry and symmetry
- [ ] rotations, invariance, and equivariance
- [ ] Cartesian tensors and their rotation laws
- [ ] decomposition into independent rotational blocks
- [ ] representations, irreps, `l`, and `m`
- [ ] spherical harmonics as Fourier-like modes on the sphere
- [ ] Wigner-D matrices and numerical equivariance
- [ ] channels versus angular components
- [ ] tensor products and Clebsch–Gordan decomposition
- [ ] higher-order coupling, `ν`, and `η_ν`
- [ ] radial basis functions and cutoffs
- [ ] MACE Eqs. 8–11
- [ ] one minimal message block and symmetry tests

**Pedagogical reference:** Sections 5.1–5.3 of *A Hitchhiker's Guide to Geometric GNNs for 3D Atomic Systems*. The prompts also preserve useful questions and analogies from your Claude discussion, but the paper and numerical checks remain the authority.

## 1. Notation ledger — write this yourself

Create a table with one row for each symbol below. Leave no symbol undefined in your eventual blog explanation.

`i`, `j`, `k`, `k̃`, `l`, `m`, `ξ`, `ν`, `L`, `M`, `η_ν`

| Symbol | Meaning |
|---|---|
| `i` | Central / receiving atom |
| `j` | Neighbor atom of `i` |
| `k` | Output feature channel of `A` or `B` |
| `k̃` | Input feature channel being mixed into `k` |
| `l` | Angular-momentum / irrep order |
| `m` | Component within order `l`; `m = -l, ..., +l` |
| `ξ` | Tensor-product factor index; one of the `ν` factors |
| `ν` | Number of `A` factors in the tensor product |
| `L` | Output angular-momentum / irrep order of `B` |
| `M` | Component within output order `L` |
| `η_ν` | Label for the coupling path that produces total order `L` |

### Index warm-up — your explanation

Answer these in your own words:

- If `l = 1`, which `m` values exist?
- For `ν = 2`, what are the two factor slots?
- What does `A[i, k̃, l_ξ, m_ξ]` select?

Blog sentence draft:

For `l = 1`, `m` takes the integer values from `-l` to `+l`: `m = -1, 0, +1`.

For `ν = 2`, there are two tensor-product factor slots: `ξ = 1` and `ξ = 2`. They contain the angular-index pairs `(l₁, m₁)` and `(l₂, m₂)`.

The entry `A[i, k̃, l_ξ, m_ξ]` selects one feature value: the feature at atom `i`, input channel `k̃`, angular order `l_ξ`, and component `m_ξ`.

## Authoring contract for a non-GDL reader

Before continuing, write down the standard every section must meet:

- define a term before abbreviating it;
- give one concrete geometric example before formal notation;
- connect the operation to something familiar from ML;
- state every tensor shape and what each axis means;
- show one visual example whenever the object has geometric meaning;
- test the claimed symmetry numerically;
- state the real/complex basis and normalization convention.

Your reader promise:

# Part I — geometric building blocks

The paper equations come later. First make each primitive visible and testable.

## 2. Atomic geometry

### Your explanation

Explain central atom `i`, neighbor `j`, relative displacement, distance, unit direction, translation invariance, and permutation-invariant neighbor pooling.

### ML bridge

Connect neighbor pooling to aggregation in an ordinary message-passing network.

### Shape ledger

### Blog paragraph

### Blog-ready explanation

We have just changed the coordinates used to describe an $l=2$ object. The next question is what happens when we rotate the *physical* direction itself. Let $R$ rotate the molecular geometry and let $\hat r$ be an edge direction. If we first rotate the direction and then evaluate its spherical harmonics, we obtain $Y^{(l)}(R\hat r)$. If we first evaluate the spherical harmonics and then rotate the resulting feature, we obtain $D^{(l)}(R)Y^{(l)}(\hat r)$. Equivariance means these two routes agree:

$$
Y^{(l)}(R\hat r)=D^{(l)}(R)Y^{(l)}(\hat r).
$$

The matrix $D^{(l)}(R)$ is called the Wigner-$D$ matrix. It acts in feature space rather than physical $x,y,z$ space, which is why its shape is $(2l+1)\times(2l+1)$. For $l=0$, it is the $1\times1$ matrix $[1]$: a scalar does not change. For $l=1$, it is a $3\times3$ rotation acting on a vector-like block. For $l=2$, it is a $5\times5$ matrix acting on the five coordinates of the symmetric-traceless tensor $Q$.

The five individual $l=2$ values are therefore not five independent scalar features. A rotation changes them and mixes them with one another, exactly as our numerical check showed. But an $l=2$ block never leaks into an $l=1$ or $l=0$ block. This is the key structural promise MACE relies on: every operation can be designed to preserve the rotation type of each block, rather than asking the network to learn rotational behaviour from data.


We consider a central carbon atom which has four hydrogen atoms. 

### Ideal tetrahedral coordinates for methane

We place the carbon atom at the origin and choose four hydrogen directions from alternating corners of a cube:

$$
(1,1,1),\quad
(1,-1,-1),\quad
(-1,1,-1),\quad
(-1,-1,1).
$$

To turn these into unit directions, divide each vector by its length:

$$
\hat{\mathbf r}
=
\frac{(\pm1,\pm1,\pm1)}{\sqrt{3}}.
$$

Any pair of these directions has dot product

$$
\hat{\mathbf r}_a \cdot \hat{\mathbf r}_b = -\frac{1}{3}.
$$

Therefore, the angle between any two C–H bonds is

$$
\theta
=
\cos^{-1}\left(-\frac{1}{3}\right)
\approx 109.47^\circ,
$$

which is the tetrahedral bond angle.

These are ideal synthetic coordinates for the geometry lab, not measured experimental coordinates.

In [3]:
pos_C = (torch.tensor((0, 0, 0))).unsqueeze(0)
sqrt_3 = torch.sqrt(torch.tensor(3))
pos_H1 = (torch.tensor((1,1,1))/sqrt_3).unsqueeze(0)
pos_H2 = (torch.tensor((1,-1,-1))/sqrt_3).unsqueeze(0)
pos_H3 = (torch.tensor((-1,1,-1))/sqrt_3).unsqueeze(0)
pos_H4 = (torch.tensor((-1,-1,1))/sqrt_3).unsqueeze(0)
pos_H2.shape
pos_C.shape
positions = torch.concat([pos_C, pos_H1, pos_H2, pos_H3, pos_H4], dim=0)

In [4]:
positions

tensor([[ 0.0000,  0.0000,  0.0000],
        [ 0.5774,  0.5774,  0.5774],
        [ 0.5774, -0.5774, -0.5774],
        [-0.5774,  0.5774, -0.5774],
        [-0.5774, -0.5774,  0.5774]])

In [5]:
# Reusable interactive 3D molecular plot
from IPython.display import HTML, display

def plot_molecule_3d(positions, labels, title="Molecule geometry"):
    # Plot atoms and bonds from a central atom to every neighbor.
    xyz = positions.detach().cpu().numpy()
    if xyz.ndim != 2 or xyz.shape[1] != 3:
        raise ValueError("positions must have shape (number_of_atoms, 3)")
    if len(labels) != len(xyz):
        raise ValueError("labels must contain one entry per atom")

    fig = go.Figure()
    for neighbor_idx in range(1, len(xyz)):
        fig.add_trace(go.Scatter3d(
            x=[xyz[0, 0], xyz[neighbor_idx, 0]],
            y=[xyz[0, 1], xyz[neighbor_idx, 1]],
            z=[xyz[0, 2], xyz[neighbor_idx, 2]],
            mode="lines",
            line=dict(color="gray", width=6),
            showlegend=False,
            hoverinfo="skip",
        ))

    fig.add_trace(go.Scatter3d(
        x=xyz[:, 0], y=xyz[:, 1], z=xyz[:, 2],
        mode="markers+text",
        text=labels,
        textposition="top center",
        hovertemplate="%{text}<br>(%{x:.3f}, %{y:.3f}, %{z:.3f})<extra></extra>",
        marker=dict(
            size=[22] + [14] * (len(xyz) - 1),
            color=["black"] + ["lightgray"] * (len(xyz) - 1),
            line=dict(color="darkslategray", width=1),
        ),
        showlegend=False,
    ))

    fig.update_layout(
        title=title,
        scene=dict(
            xaxis_title="x", yaxis_title="y", zaxis_title="z",
            aspectmode="data",
        ),
        margin=dict(l=0, r=0, b=0, t=45),
    )
    return fig

fig = plot_molecule_3d(
    positions,
    labels=["C", "H1", "H2", "H3", "H4"],
    title="Ideal methane geometry",
)
display(HTML(fig.to_html(full_html=False, include_plotlyjs="cdn")))

In [6]:
torch.manual_seed(42)
translation_vector = 5*torch.rand(1, 3)
new_positions = positions + translation_vector
neighbor_order = torch.randperm(4) + 1
indices = torch.cat([torch.tensor([0]), neighbor_order])
new_pos_perm = new_positions[indices]
labels=["C", "H1", "H2", "H3", "H4"]
new_labels_perm = [labels[i] for i in indices.tolist()]
# new_pos_perm
fig = plot_molecule_3d(
    new_pos_perm,
    labels=new_labels_perm,
    title="Ideal methane geometry",
)
display(HTML(fig.to_html(full_html=False, include_plotlyjs="cdn")))

When we rotate the entire molecule, every position vector changes direction, but all pairwise distances remain unchanged; distances are rotationally invariant.

In [7]:
print("original order:", labels)
print("permuted order:", new_labels_perm)
print("row indices:", indices.tolist())

original order: ['C', 'H1', 'H2', 'H3', 'H4']
permuted order: ['C', 'H3', 'H2', 'H4', 'H1']
row indices: [0, 3, 2, 4, 1]


In [8]:
print(positions - positions[0])


tensor([[ 0.0000,  0.0000,  0.0000],
        [ 0.5774,  0.5774,  0.5774],
        [ 0.5774, -0.5774, -0.5774],
        [-0.5774,  0.5774, -0.5774],
        [-0.5774, -0.5774,  0.5774]])


In [9]:
new_positions - new_positions[0]

tensor([[ 0.0000,  0.0000,  0.0000],
        [ 0.5774,  0.5774,  0.5774],
        [ 0.5774, -0.5774, -0.5774],
        [-0.5774,  0.5774, -0.5774],
        [-0.5774, -0.5774,  0.5774]])

In [10]:
# LAB 2 — atomic geometry
# TODO (you): create one central atom and at least three neighbors in 3D.
# TODO (you): compute relative displacements, distances, and unit directions.
# TODO (you): make a labeled 3D scatter/quiver plot.
# TODO (you): translate all atoms and permute neighbor order.
# PASS: relative geometry survives translation and pooled output survives permutation.

## 3. Rotations, invariance, and equivariance

### Your explanation

Using one rotated vector, explain a rotation matrix, an invariant quantity, an equivariant quantity, and the two routes in an equivariance check.

### ML bridge

Compare equivariance with weight sharing in convolution, without claiming they are identical.

### Blog paragraph

3D rotation matrices are basically numbers which help us rotate a vector around an axis. Let us use a simple rotation to demonstrate, such as a 90° rotation on the Z-axis. 

The rotation matrix for this is of the form
[
    cos(theta) 0 0
    0 sin(theta) 0
    0 0 1
]

In [11]:
positions

tensor([[ 0.0000,  0.0000,  0.0000],
        [ 0.5774,  0.5774,  0.5774],
        [ 0.5774, -0.5774, -0.5774],
        [-0.5774,  0.5774, -0.5774],
        [-0.5774, -0.5774,  0.5774]])

In [12]:
torch.tensor([[1,2],[3,4]])

tensor([[1, 2],
        [3, 4]])

In [13]:
def rotate_z_axis(positions, theta):
    theta = torch.tensor(theta)
    c = torch.cos(theta)
    s = torch.sin(theta)
    rotation_matrix = torch.tensor(
        [
            [c, -s, 0],
            [s, c, 0],
            [0,0,1]
        ]
        )
    rotated_positions = positions @ rotation_matrix.T 
    return rotation_matrix, rotated_positions
rotation_matrix, rotated_positions = rotate_z_axis(positions, math.pi/2)

In [14]:
print(positions)
print(rotated_positions)

tensor([[ 0.0000,  0.0000,  0.0000],
        [ 0.5774,  0.5774,  0.5774],
        [ 0.5774, -0.5774, -0.5774],
        [-0.5774,  0.5774, -0.5774],
        [-0.5774, -0.5774,  0.5774]])
tensor([[ 0.0000,  0.0000,  0.0000],
        [-0.5774,  0.5774,  0.5774],
        [ 0.5774,  0.5774, -0.5774],
        [-0.5774, -0.5774, -0.5774],
        [ 0.5774, -0.5774,  0.5774]])


In [15]:
fig = plot_molecule_3d(
    rotated_positions,
    labels=new_labels_perm,
    title="Ideal methane geometry",
)
display(HTML(fig.to_html(full_html=False, include_plotlyjs="cdn")))

In [16]:
print(rotation_matrix.T @ rotation_matrix)
print(torch.linalg.det(rotation_matrix))

tensor([[1., 0., 0.],
        [0., 1., 0.],
        [0., 0., 1.]])
tensor(1.)


R transpose R = identity, which means the rotation preserves dot products. Preserving dot products means that lengths and angles are preserved, so R transpose is the inverse rotation, and R is these two lengths and vectors. 

In [17]:
rot_vec1, new_position_1 = rotate_z_axis(positions, math.pi/4)
rot_vec2, new_position_2 = rotate_z_axis(new_position_1, math.pi/3)
print(new_position_2)
new_position_3 = positions @ (rot_vec2 @ rot_vec1).T
print(new_position_3)


tensor([[ 0.0000,  0.0000,  0.0000],
        [-0.7071,  0.4082,  0.5774],
        [ 0.4082,  0.7071, -0.5774],
        [-0.4082, -0.7071, -0.5774],
        [ 0.7071, -0.4082,  0.5774]])
tensor([[ 0.0000,  0.0000,  0.0000],
        [-0.7071,  0.4082,  0.5774],
        [ 0.4082,  0.7071, -0.5774],
        [-0.4082, -0.7071, -0.5774],
        [ 0.7071, -0.4082,  0.5774]])


Again, in the cell above, I have verified that rotating the atom once and then rotating it again, and then doing the composite rotation, amounts to the same thing. As we can see, the two matrices are exactly the same. 

In [18]:
# LAB 3 — rotations
# TODO (you): construct a valid 3D rotation matrix from an axis and angle.
# TODO (you): plot a toy neighborhood before and after rotation.
# TODO (you): track one invariant scalar and one equivariant vector.
# CHECK: RᵀR = I, det(R) = +1, and vector norms are preserved.
# CHECK: composing two rotations agrees with sequential application.

## 4. Cartesian tensors before irreps

Do not use the word *irrep* yet. Begin with objects an ML practitioner already knows: a scalar, a 3D vector, and a 3×3 matrix.

### Your explanation

Starting with two vectors `v` and `w`, explain the outer product `T = v ⊗ w`, why it has shape `(3, 3)`, and why it contains nine numbers rather than three. Then derive its rotation law from `v' = Rv` and `w' = Rw`.

### Shape and transformation ledger

Record the shape and rotation law of a scalar, a vector, and this matrix. Explicitly distinguish **tensor rank** from the number of stored components.

### ML bridge

Relate the outer product to multiplicative feature interactions, but explain why geometry dictates how the result must transform.

### Blog paragraph

Let us begin with the simplest operation that combines two vectors without throwing away their directional information. Consider

$$
v = (1,2,5), \qquad w = (6,7,17).
$$

The **outer product** (also called a rank-2 tensor product) forms every pairwise product between one component of `v` and one component of `w`. We write it as $T = v \otimes w$. Its entries are

$$
T_{ij} = v_i w_j,
$$

so two vectors with shape `(3,)` produce a matrix with shape `(3, 3)`, containing $3 \times 3 = 9$ values:

$$
T =
\begin{bmatrix}
v_1w_1 & v_1w_2 & v_1w_3 \\
v_2w_1 & v_2w_2 & v_2w_3 \\
v_3w_1 & v_3w_2 & v_3w_3
\end{bmatrix}.
$$

This is different from a dot product. The dot product compresses two vectors into one scalar, while the outer product preserves every pairwise interaction. That makes it useful when the relationship between directions matters, for example in second-order geometric features and multiplicative feature interactions in machine learning.

The operation itself has no learned parameters; it is just multiplication and arrangement. Its geometric importance comes from its transformation law. If both vectors are rotated by `R`, then

$$
(Rv) \otimes (Rw) = R T R^\top.
$$

Thus `T` is a Cartesian rank-2 tensor. The nine entries are not yet nine unrelated feature types: this nine-dimensional object is reducible and will later split into the `1 + 3 + 5` rotational blocks.

In [19]:
v = torch.tensor([1., 2., 5.])
w = torch.tensor([6., 7., 17.])
T = torch.outer(v, w)
T

tensor([[ 6.,  7., 17.],
        [12., 14., 34.],
        [30., 35., 85.]])

Next, we'll rotate each vector v and w individually and find the tensor product on the rotated vectors. We want to verify the following identity. 

\((Rv)\otimes(Rw)=RTR^\top.\)

Let's choose the rotation matrix for 45° around z axis. 
So, 
R = [cospi/4 -sinpi/4 0 

    sinpi/4.  cospi/4 0
    
    0 0 1]

In [20]:
c = torch.cos(torch.tensor(torch.pi/4))
s = torch.sin(torch.tensor(torch.pi/4))
R = torch.tensor([[c,-s,0], [s,c,0], [0,0,1]])
rot_v = R @ (v.reshape(-1,1))
rot_w = R @ (w.reshape(-1,1))
rot_T = torch.outer(rot_v.reshape(-1), rot_w.reshape(-1))
print(rot_T)

print(R @ T @ R.T)




tensor([[  0.5000,  -6.5000, -12.0208],
        [ -1.5000,  19.5000,  36.0624],
        [ -3.5355,  45.9619,  85.0000]])
tensor([[  0.5000,  -6.5000, -12.0208],
        [ -1.5000,  19.5000,  36.0624],
        [ -3.5355,  45.9619,  85.0000]])


The outer product is equivariant: rotating both input vectors and then forming their outer product gives the same result as rotating the resulting rank-2 tensor with \(RTR^\top\).

In [21]:
# LAB 4 — a Cartesian rank-2 tensor
# TODO (you): choose two non-parallel 3D vectors v and w.
# TODO (you): form T = v ⊗ w and print both its 3×3 and flat 9-value forms.
# TODO (you): rotate v and w, then form T_rotated_from_vectors.
# TODO (you): separately transform T using R @ T @ R.T.
# CHECK: the two transformed tensors agree, including their shapes.

## 5. Discover the independent rotational pieces

The nine matrix entries do not all behave as one indivisible object. Discover the smaller families before naming them.

### Your explanation

Split a generic 3×3 matrix into:

1. its trace part;
2. its antisymmetric part;
3. its symmetric-traceless part.

Count their independent numbers and explain, from the rotation experiment, what it means for each family to remain closed under rotation. Use the concrete statement `9 = 1 + 3 + 5` before introducing any angular-order notation.

### Naming checkpoint

Only after the experiment, write your own definition of an **irreducible representation**. Test the phrase *an independent rotational block*: its components may mix with one another, but rotation does not leak them into another block. Where is this analogy useful, and where might it be misleading?

### Blog paragraph

The outer product gave us a 3×3 matrix with nine entries. We now look for smaller families of entries that have their own consistent symmetry under rotation. We will first call them **pieces**; we will reserve the formal word *irreducible* until after we have counted and tested them.

The first piece is the symmetric part of `T`, defined by

$$
T_{\mathrm{sym}}=\frac{T+T^\top}{2}.
$$

In [22]:
T_sym = (T + T.T)/2
T_sym

tensor([[ 6.0000,  9.5000, 23.5000],
        [ 9.5000, 14.0000, 34.5000],
        [23.5000, 34.5000, 85.0000]])

Because `T_sym.T = T_sym`, its entries mirror across the main diagonal: the upper- and lower-triangular parts match. It has three independent diagonal entries and three independent off-diagonal entries.

The second piece is the antisymmetric part:

$$
T_{\mathrm{anti}}=\frac{T-T^\top}{2}.
$$

In [23]:
T_anti = (T - T.T)/2

T_anti

tensor([[ 0.0000, -2.5000, -6.5000],
        [ 2.5000,  0.0000, -0.5000],
        [ 6.5000,  0.5000,  0.0000]])

For an antisymmetric matrix, `T_anti.T = -T_anti`. Therefore its diagonal is zero, and each lower-triangular entry is the negative of its matching upper-triangular entry. Only three entries are independent.

We now remove the trace from the symmetric part to obtain the symmetric-traceless piece.

In [24]:
tr_T = torch.trace(T)
T_ST = T_sym - tr_T/3 * torch.eye(3)
T_ST

tensor([[-29.0000,   9.5000,  23.5000],
        [  9.5000, -21.0000,  34.5000],
        [ 23.5000,  34.5000,  50.0000]])

The remaining trace piece is the isotropic part of the symmetric matrix. It has one scalar degree of freedom and is represented by

$$
T_{\mathrm{tr}}=\frac{\operatorname{tr}(T)}{3}I.
$$

The factor of `1/3` distributes the total trace equally across the three diagonal entries.

In [25]:
T_tr = tr_T/3 * torch.eye(3)
T_tr

tensor([[35.,  0.,  0.],
        [ 0., 35.,  0.],
        [ 0.,  0., 35.]])

We now have all three pieces of the original tensor: the trace, antisymmetric, and symmetric-traceless parts. They reconstruct the original matrix:

$$
T=T_{\mathrm{tr}}+T_{\mathrm{anti}}+T_{\mathrm{ST}}.
$$

The following code checks reconstruction, zero trace, and the symmetry properties.

In [26]:
T_reconstructed = T_tr + T_anti + T_ST
T_reconstructed

print(torch.allclose(T_reconstructed, T))
print(torch.trace(T_ST))
print(torch.allclose(T_ST, T_ST.T))

True
tensor(0.)
True


### Counting the independent numbers

We can now identify exactly which numbers are free to choose.

The trace piece contains one number, `t`:

$$
T_{\mathrm{tr}}=\begin{bmatrix}t&0&0\\0&t&0\\0&0&t\end{bmatrix}.
$$

The antisymmetric piece contains three independent upper-triangular values, `a`, `b`, and `c`; the other entries are forced:

$$
T_{\mathrm{anti}}=\begin{bmatrix}0&a&b\\-a&0&c\\-b&-c&0\end{bmatrix}.
$$

The symmetric-traceless piece contains three independent off-diagonal values and two independent diagonal values:

$$
T_{\mathrm{ST}}=\begin{bmatrix}p&q&r\\q&s&u\\r&u&-p-s\end{bmatrix}.
$$

The final diagonal entry is forced because the diagonal entries must sum to zero. Therefore the count is

$$
9=\underbrace{1}_{\text{trace}}+\underbrace{3}_{\text{antisymmetric}}+\underbrace{5}_{\text{symmetric-traceless}}.
$$

At this point, the count only shows that the three pieces retain all nine degrees of freedom. It does not yet prove that each piece remains closed under rotation; that is the next experiment.

Counting the pieces is not enough: we must check how they behave under rotation. We will compare two routes. In route A, rotate `T` and then decompose the rotated tensor. In route B, decompose `T` first and rotate each piece separately using $RXR^\top$. A matching total sum checks reconstruction, but the key test is that each corresponding pair of pieces matches individually. If it does, rotation does not cause information to leak from one family into another.

In [27]:
T_rot = R @ T @ R.T
T_rot

tensor([[  0.5000,  -6.5000, -12.0208],
        [ -1.5000,  19.5000,  36.0624],
        [ -3.5355,  45.9619,  85.0000]])

Let us rotate T_tr first \(T_{\mathrm{tr}}' \stackrel{?}{=} R\,T_{\mathrm{tr}}\,R^\top.\)

In [28]:
T_tr_rot = R @ T_tr @ R.T
T_ST_rot = R @ T_ST @ R.T
T_anti_rot = R @ T_anti @ R.T
print(T_tr_rot)
print(T_anti_rot)
print(T_ST_rot)


tensor([[35.,  0.,  0.],
        [ 0., 35.,  0.],
        [ 0.,  0., 35.]])
tensor([[ 0.0000, -2.5000, -4.2426],
        [ 2.5000,  0.0000, -4.9497],
        [ 4.2426,  4.9497,  0.0000]])
tensor([[-34.5000,  -4.0000,  -7.7782],
        [ -4.0000, -15.5000,  41.0122],
        [ -7.7782,  41.0122,  50.0000]])


In [29]:
print(T_tr_rot + T_ST_rot + T_anti_rot)
print(T_rot)

tensor([[  0.5000,  -6.5000, -12.0208],
        [ -1.5000,  19.5000,  36.0624],
        [ -3.5355,  45.9619,  85.0000]])
tensor([[  0.5000,  -6.5000, -12.0208],
        [ -1.5000,  19.5000,  36.0624],
        [ -3.5355,  45.9619,  85.0000]])


### Decompose the rotated tensor

Apply exactly the same trace, symmetric, antisymmetric, and symmetric-traceless formulas to `T_rot`. Do not introduce a new decomposition rule merely because the tensor has been rotated.

In [30]:
T_rot_sym = (T_rot + T_rot.T) / 2
tr_T_rot = torch.trace(T_rot)
T_rot_tr = tr_T_rot / 3 * torch.eye(3)
T_rot_ST = T_rot_sym - T_rot_tr
T_rot_anti = (T_rot - T_rot.T) / 2

print("trace closed:", torch.allclose(T_rot_tr, T_tr_rot))
print("antisymmetric closed:", torch.allclose(T_rot_anti, T_anti_rot))
print("symmetric-traceless closed:", torch.allclose(T_rot_ST, T_ST_rot))

trace closed: True
antisymmetric closed: True
symmetric-traceless closed: True


If all three checks print `True`, each piece follows the same rotation law on both routes. The trace, antisymmetric, and symmetric-traceless families are therefore closed under rotations: a rotation can mix components *within* one family, but cannot move information into a different family. This is the concrete observation that motivates the irreducible-representation language in the next section.

In [31]:
# LAB 5 — discover 9 = 1 + 3 + 5
# TODO (you): split a generic 3×3 tensor into trace, antisymmetric, and
# symmetric-traceless pieces.
# TODO (you): count the independent values in each piece.
# TODO (you): reconstruct the original tensor from the three pieces.
# TODO (you): compare rotate-then-decompose with decompose-then-rotate.
# CHECK: dimensions are 1 + 3 + 5 = 9 and both rotation routes agree.
# CHECK: trace never leaks into the other pieces, and vice versa.

## 6. Representations and irreducible representations

The experiment above motivates the formal language. A **representation** is a rule, written `D(R)`, that tells us how a feature's components change when the physical system is rotated by `R`. If a feature space separates into smaller subspaces that never mix with one another under rotation, those subspaces are independent rotational blocks. An **irreducible representation**, or **irrep**, is a block that cannot be split any further while preserving that property.

For three-dimensional rotations, irreps are labelled by a non-negative integer `l`. This `l` is a label for the rotational type; it is not the matrix rank and it is not a component index. A block of order `l` always contains `2l + 1` components.

The symbol `m` labels a component *inside one fixed-`l` block*. It runs through

$$
m=-l,-l+1,\ldots,0,\ldots,l-1,l.
$$

Therefore the three blocks discovered above are named

| Cartesian piece | Irrep label | Number of components | Component labels |
|---|---:|---:|---|
| trace | $l=0$ | $1$ | $m=0$ |
| antisymmetric / vector-like | $l=1$ | $3$ | $m=-1,0,+1$ |
| symmetric-traceless | $l=2$ | $5$ | $m=-2,-1,0,+1,+2$ |

The free Cartesian entries we used earlier are not yet the `m`-components. The `m` labels refer to coordinates in a spherical basis. A later change of basis will connect the Cartesian entries to those spherical components. Under rotation, the `m`-components within one `l` block generally mix with one another, but they do not mix with a different `l` block.

### Your explanation

Now attach formal names to the experiment. Explain what a representation `D(R)` acts on, why the same physical rotation uses a 1×1, 3×3, or 5×5 matrix on the three blocks above, and why `D(R₂R₁) = D(R₂)D(R₁)` is the feature-space version of your earlier rotation-composition check.

### ML bridge

Relate block structure to keeping different feature types in separate tensor blocks. A representation is the rule for how a feature vector changes; an irrep is a block that cannot be split into smaller rotation-closed blocks.

### Blog paragraph

The next cell is your original first draft. Preserve it, but revisit it only after completing Labs 4–6; annotate what changed in your understanding rather than silently replacing the draft.

The D matrix is the Wagner D matrix for different orders of irreducible order. These are tensors of different size, so they require different rotational tensors. For example, for L = 0, that's a scalar rotation, which makes no difference at all, so it's just a scalar, 1, scaling by 1 for a vector. You usually have a 2-by-2 matrix or, depending on the number of dimensions, a 3-by-3 matrix. For higher-order tensors, you have higher-order matrices. You have these angular breakdowns, etc. 

### Blog-ready explanation

A rotation does not act on every feature in the same way. A scalar, such as a temperature or an atom’s identity embedding, is unchanged, so its rotation rule is the one-by-one matrix $D^0(R)=[1]$. A three-dimensional vector does change, and its three components transform with the ordinary rotation matrix $D^1(R)=R$. The matrix $D^l(R)$ is therefore not one universal matrix: its size depends on the rotational type of the feature it acts on.

The nine entries of a $3\times3$ tensor give a useful example. They can be reorganised into three independent rotational blocks: a one-component trace ($l=0$), a three-component antisymmetric or vector-like part ($l=1$), and a five-component symmetric-traceless part ($l=2$). Each block has its own transformation matrix: $1\times1$, $3\times3$, or $5\times5$. The blocks may mix components internally, but a rotation cannot send information from one block into another.

This is what the word *irreducible* means here: the block is a smallest unit that remains closed under every rotation. The label $l$ identifies the block, while $m=-l,\ldots,l$ labels the individual components inside it. Thus $l$ tells us *which kind of rotational object* we have, and $m$ tells us *which coordinate within that object* we are looking at.

Finally, the representation matrices obey $D(R_2R_1)=D(R_2)D(R_1)$. This is simply the familiar rule that two physical rotations applied in sequence can be represented by multiplying their feature-space matrices in the same order. The MACE architecture uses these blocks so that neural-network operations can learn from features while preserving their precise behaviour under rotation.

In [32]:
R1, _ = rotate_z_axis(torch.eye(3), math.pi/6)
R2, _ = rotate_z_axis(torch.eye(3), math.pi/12)

In [33]:
print(R1)
print(R2)

tensor([[ 0.8660, -0.5000,  0.0000],
        [ 0.5000,  0.8660,  0.0000],
        [ 0.0000,  0.0000,  1.0000]])
tensor([[ 0.9659, -0.2588,  0.0000],
        [ 0.2588,  0.9659,  0.0000],
        [ 0.0000,  0.0000,  1.0000]])


In [34]:
def D0(R):
    return torch.tensor([1])
def D1(R):
    return R

D0_R1 = D0(R1)
D0_R2 = D0(R2)
D1_R1 = D1(R1)
D1_R2 = D1(R2)


print(torch.allclose( (D1_R2 @ D1_R1), D1(R2 @ R1)) )
print(torch.allclose( (D0_R2 @ D0_R1), D0(R2 @ R1)) )

True
True


In [35]:
# LAB 6 — first representations
# TODO (you): represent the same rotation on a scalar and a 3D vector.
# TODO (you): record the shapes of D⁰(R), D¹(R), and D²(R).
# TODO (you): connect those blocks to the 1, 3, and 5 pieces in Lab 5.
# CHECK: D(R₂R₁) = D(R₂)D(R₁) for every feature type.

## 7. Angular orders `l` and components `m` in detail

### Your explanation

Build on the definitions above without assuming quantum mechanics. Explain the component count `2l + 1`, the allowed `m` values, why components with different `m` mix under rotation while `l` remains fixed, and why `(l_ξ, m_ξ)` denotes a pair belonging to factor `ξ` rather than equal values.

Create a table for `l = 0, 1, 2, 3` containing component count, allowed `m`, and geometric intuition.

### Blog paragraph

Once we have identified the rotational blocks, we need a compact way to name them. The integer $l$ labels the irreducible representation, or angular order; it is not one of the components inside that representation. Each fixed-$l$ block contains $2l+1$ components, indexed by $m=-l,\ldots,+l$. Thus $l=0$ has one component, $l=1$ has three, $l=2$ has five, and $l=3$ has seven.

Under a rotation, the components with different $m$ values may mix with one another, but the value of $l$ remains fixed. An $l=2$ feature can therefore rotate within its five-dimensional space, but it cannot turn into an $l=1$ or $l=3$ feature. For now, we are focusing only on this rotational bookkeeping; feature channels will be introduced later.

In [36]:
list(range(-2,2))

[-2, -1, 0, 1]

In [37]:
def valid_l_m(max_order=3):
    l = range(max_order + 1)
    m = {i:list(range(-i, i+1)) for i in l}
    return l,m
l,m = valid_l_m(3)
print(l)
print(m)

range(0, 4)
{0: [0], 1: [-1, 0, 1], 2: [-2, -1, 0, 1, 2], 3: [-3, -2, -1, 0, 1, 2, 3]}


In [38]:
for i in l:
    assert len(m[i]) == 2*i+1

In [39]:
# LAB 7 — angular index structure
# TODO (you): generate valid m values for several l values.
# TODO (you): draw a simple feature-layout diagram showing each fixed-l block and its m components.
# CHECK: every l block contains exactly 2l + 1 components.

## 8. Spherical harmonics — Fourier-like modes on the sphere

### Your explanation

Begin by connecting this section to the tensor experiment: the $3\times3$ tensor was decomposed into $l=0,1,2$ rotational blocks, while spherical harmonics provide a convenient direction-dependent basis for features living in those same kinds of blocks. Explain spherical harmonics as fixed direction-dependent basis functions and as a Fourier-like basis for functions on the sphere. Explain their input/output, the visual role of `l` and `m`, why distance is absent, why all `m` values for one `l` form a rotating block, and which real-basis normalization you use.

Use the `l = 1` real harmonics as the first bridge: check numerically whether they are proportional to the unit-direction components `x`, `y`, and `z` under your chosen ordering and normalization. Do not memorize the ordering; document it.

### ML bridge

Compare them with a fixed feature map or positional encoding while explaining the extra rotational structure. Then compare `l` with Fourier frequency and `m` with the components inside one frequency block; record both the power and the limits of this analogy.

### Blog paragraph

### Blog-ready explanation

At first, spherical harmonics may look like a completely new topic. They are actually the bridge from our Cartesian tensor experiment to the representation used by MACE. The tensor $T$ showed that rotational information naturally separates into blocks with $l=0$, $l=1$, and $l=2$. Spherical harmonics give us a convenient spherical basis for direction-dependent features in those same rotational types. We are not decomposing the same numerical tensor again; we are changing from Cartesian coordinates to a basis whose rotation law is already organised by $l$ and $m$.

A spherical harmonic is a fixed function on the surface of a sphere. It takes a direction, written as the unit vector $\hat{r}=r/\lVert r\rVert$, and returns a number describing how strongly that direction matches one particular angular pattern. For a fixed angular order $l$, we evaluate all $2l+1$ components, indexed by $m$, and treat them as one rotational feature block.

The order $l$ controls the angular complexity of the pattern. The $l=0$ harmonic is constant over the sphere, so it carries no directional detail. The $l=1$ harmonics are the simplest directional patterns; in a suitable real basis they are proportional to the three familiar coordinates $x$, $y$, and $z$ of a unit vector. Higher orders describe progressively finer angular variation. The exact ordering and signs depend on the basis convention, so we will document the convention rather than rely on memorisation.

Spherical harmonics deliberately ignore distance. That is useful because a molecular edge has two different kinds of information: *how far apart* two atoms are and *which way* one atom lies from the other. MACE handles the first with radial basis functions and the second with spherical harmonics. Keeping these roles separate gives the model a clean radial-versus-angular representation.

The important symmetry fact is that rotating the input direction does not destroy the feature; it rotates the whole fixed-$l$ block in a predictable way. With our convention, this is written as $Y^l(R\hat{r})=D^l(R)Y^l(\hat{r})$. The spherical harmonics provide the direction-dependent features, while the Wigner-$D$ matrix tells us how those features mix when the molecule is rotated. In machine-learning language, they are a structured, symmetry-aware positional encoding on the sphere—not arbitrary coordinates that would have to be relearned for every orientation.

### The missing bridge: from a Cartesian tensor to an $l=2$ block

Before using spherical harmonics, let us connect them carefully to the tensor decomposition we have already built. A tensor is not merely an array of numbers; it is an array together with a rule for how those numbers change under a transformation. In MACE the transformations of interest are three-dimensional rotations, so both the Cartesian and spherical descriptions below are representations of the same group, $SO(3)$. They organise the coordinates differently, but they do not describe different physics.

A Cartesian rank-two tensor $T$ is a $3\times3$ matrix that transforms as

$$
T \longmapsto RTR^{\mathsf T}.
$$

Its nine entries can be separated into three pieces that rotations never mix with one another:

$$
T_{\mathrm{tr}}=\frac{\operatorname{tr}(T)}{3}I,
\qquad
T_{\mathrm{anti}}=\frac{T-T^{\mathsf T}}{2},
\qquad
T_{\mathrm{ST}}=\frac{T+T^{\mathsf T}}{2}-T_{\mathrm{tr}}.
$$

The trace piece needs one independent number, the antisymmetric piece needs three, and the symmetric-traceless piece needs five. Therefore

$$
9=1+3+5.
$$

These three closed rotation types are called $l=0$, $l=1$, and $l=2$. Here $l$ is **not** the rank of the original Cartesian tensor. It labels an irreducible rotation type. An $l$ block always contains $2l+1$ components, and $m$ indexes the components inside that block. A rotation may mix the different $m$ components belonging to one $l$, but it cannot mix components belonging to different $l$ values.

Now begin with a molecular edge direction

$$
\hat r=(x,y,z),\qquad x^2+y^2+z^2=1,
$$

and form its outer product with itself:

$$
T=\hat r\hat r^{\mathsf T}
=
\begin{bmatrix}
x^2 & xy & xz\\
xy & y^2 & yz\\
xz & yz & z^2
\end{bmatrix}.
$$

This special $T$ is already symmetric, so $T_{\mathrm{anti}}=0$. Its trace is the scalar number $1$, but its **trace matrix piece** is $T_{\mathrm{tr}}=I/3$. The remaining information is

$$
T_{\mathrm{ST}}=\hat r\hat r^{\mathsf T}-\frac{I}{3}
=
\begin{bmatrix}
x^2-\frac13 & xy & xz\\
xy & y^2-\frac13 & yz\\
xz & yz & z^2-\frac13
\end{bmatrix}.
$$

This is the $l=2$ part of the directional outer product. It is also commonly called the **quadrupole tensor** and denoted $Q$; in this section, $Q$ is only shorthand for the object we already know: $Q:=T_{\mathrm{ST}}$.

Although the matrix displays nine entries, symmetry and the zero-trace condition leave only five independent numbers. One simple real Cartesian coordinate list is

$$
q_{\mathrm{cart}}=
(T_{xx},T_{yy},T_{xy},T_{xz},T_{yz}),
$$

where all entries on the right refer to $T_{\mathrm{ST}}$. The missing diagonal entry is forced to be $T_{zz}=-(T_{xx}+T_{yy})$. Thus the five numbers $(a,b,d,e,f)$ reconstruct the entire matrix as

$$
T_{\mathrm{ST}}=
\begin{bmatrix}
a & d & e\\
d & b & f\\
e & f & -a-b
\end{bmatrix}.
$$

This explains the connection to spherical harmonics. The operation $T\mapsto(T_{\mathrm{tr}},T_{\mathrm{anti}},T_{\mathrm{ST}})$ discovers the irreducible rotation types already present inside a Cartesian matrix. In contrast, the map

$$
\hat r\longmapsto Y^{(l)}(\hat r)
$$

directly manufactures a directional feature of a chosen rotation type. Up to normalization and a fixed change of coordinates,

$$
Y^{(0)}(\hat r)\sim 1,
\qquad
Y^{(1)}(\hat r)\sim\hat r,
\qquad
Y^{(2)}(\hat r)\sim\operatorname{coordinates}\!\left(\hat r\hat r^{\mathsf T}-\frac{I}{3}\right).
$$

So the five entries returned by a real $l=2$ spherical-harmonics routine contain the same *kind* of rotational information as the five independent entries of $T_{\mathrm{ST}}$. They are not usually the same five raw numbers: e3nn uses a fixed orthogonal, normalized real spherical basis, related to our Cartesian list by a constant change-of-basis matrix.

#### Where the complex basis fits

MACE and e3nn use real spherical harmonics, so no complex arithmetic is required here. The complex basis becomes pedagogically useful later, before Wigner-$D$ matrices and Clebsch–Gordan products, because rotations about the $z$ axis act especially simply on combinations such as $x\pm iy$. It clarifies the conventional labels $m=-l,\ldots,l$ and the selection rule $m_1+m_2=M$. The real and complex bases have the same expressive power; the real basis is convenient for real molecular data and avoids complex-valued storage and arithmetic.


In [40]:
# LAB — derive the five independent l=2 coordinates from one direction.
r_hat = torch.tensor([1 / math.sqrt(2), 1 / math.sqrt(2), 0.0])
T_direction = torch.outer(r_hat, r_hat)
T_trace = torch.trace(T_direction) / 3 * torch.eye(3)
T_anti = (T_direction - T_direction.T) / 2
T_st = (T_direction + T_direction.T) / 2 - T_trace

l2_cartesian = torch.stack(
    [T_st[0, 0], T_st[1, 1], T_st[0, 1], T_st[0, 2], T_st[1, 2]]
)
a, b, d, e, f = l2_cartesian
T_st_reconstructed = torch.stack(
    [
        torch.stack([a, d, e]),
        torch.stack([d, b, f]),
        torch.stack([e, f, -a - b]),
    ]
)

assert torch.allclose(T_direction, T_trace + T_anti + T_st)
assert torch.allclose(T_anti, torch.zeros_like(T_anti))
assert torch.allclose(torch.trace(T_st), torch.tensor(0.0), atol=1e-6)
assert torch.allclose(T_st, T_st_reconstructed)

print("T = r_hat r_hat^T:\n", T_direction)
print("\nT_trace:\n", T_trace)
print("\nT_anti:\n", T_anti)
print("\nT_ST:\n", T_st)
print("\nFive independent Cartesian l=2 coordinates:\n", l2_cartesian)


T = r_hat r_hat^T:
 tensor([[0.5000, 0.5000, 0.0000],
        [0.5000, 0.5000, 0.0000],
        [0.0000, 0.0000, 0.0000]])

T_trace:
 tensor([[0.3333, 0.0000, 0.0000],
        [0.0000, 0.3333, 0.0000],
        [0.0000, 0.0000, 0.3333]])

T_anti:
 tensor([[0., 0., 0.],
        [0., 0., 0.],
        [0., 0., 0.]])

T_ST:
 tensor([[ 0.1667,  0.5000,  0.0000],
        [ 0.5000,  0.1667,  0.0000],
        [ 0.0000,  0.0000, -0.3333]])

Five independent Cartesian l=2 coordinates:
 tensor([0.1667, 0.1667, 0.5000, 0.0000, 0.0000])


In [41]:
# Reused from papers/14-RFM/src/geometry.py.
def sample_sphere(n: int, device: torch.device | str | None = None) -> torch.Tensor:
    """Return n approximately uniform points on the unit sphere."""
    if n <= 0:
        raise ValueError("n must be positive")
    z = 2 * torch.rand(n, 1, device=device) - 1
    r = torch.sqrt(1 - torch.square(z))
    theta = 2 * torch.pi * torch.rand(n, 1, device=device) - 1
    x = (r * torch.cos(theta)).reshape(-1, 1)
    y = (r * torch.sin(theta)).reshape(-1, 1)
    return torch.concat([x, y, z], dim=1)

In [42]:
n = 10
x= sample_sphere(n)
print(x.shape)
print(torch.linalg.vector_norm(x, dim=1))


torch.Size([10, 3])
tensor([1.0000, 1.0000, 1.0000, 1.0000, 1.0000, 1.0000, 1.0000, 1.0000, 1.0000,
        1.0000])


In [43]:
sqrt_pi = torch.sqrt(torch.tensor(torch.pi))
Y_0_0 = (0.5 / sqrt_pi).repeat(n)

In [44]:
Y_0_0

tensor([0.2821, 0.2821, 0.2821, 0.2821, 0.2821, 0.2821, 0.2821, 0.2821, 0.2821,
        0.2821])

In [45]:
assert Y_0_0.shape == (n,)
assert torch.allclose(Y_0_0, Y_0_0[0].expand_as(Y_0_0))

fig = go.Figure(
    go.Scatter3d(
        x=x[:, 0].tolist(),
        y=x[:, 1].tolist(),
        z=x[:, 2].tolist(),
        mode="markers",
        marker={
            "size": 6,
            "color": Y_0_0.tolist(),
            "colorscale": "Viridis",
            "showscale": True,
            "colorbar": {"title": "Y₀⁰"},
        },
    )
)
fig.update_layout(
    title="l=0, m=0: constant spherical harmonic",
    scene={
        "xaxis_title": "x",
        "yaxis_title": "y",
        "zaxis_title": "z",
        "aspectmode": "data",
    },
)
display(HTML(fig.to_html(full_html=False, include_plotlyjs="cdn")))

### Using a standard spherical-harmonics implementation

At this point, we stop hand-writing increasingly complicated polynomial formulas. The formulas are useful for seeing the pattern, but they are not the right way to build a reliable implementation. Instead, we use the `e3nn` library, which evaluates the real spherical harmonics for us in a PyTorch-friendly representation.

The input is a batch of directions with shape `(n, 3)`. For a chosen order `l`, the routine returns a tensor with shape `(n, 2l+1)`: one row for each direction and one column for each component in that rotational block. With `normalize=True`, the input directions are treated as points on the unit sphere. With `normalization="integral"`, the output uses the same unit-integral convention as our earlier value $Y_0^0=1/\sqrt{4\pi}$.

This is not abandoning the mathematics. The earlier formulas tell us what kind of object the routine must produce: an `l=0` scalar block, an `l=1` vector-like block, an `l=2` five-component block, and so on. The library simply performs the careful polynomial evaluation and normalization. We still check the shapes, inspect the plots, and record the library's component ordering before calling those components `m` values.

The useful visual is a Fourier-style circle rather than a cloud of 3D sample points. We freeze the polar angle at the equator and let the direction travel once around the circle. Plotting each component against the azimuth shows the idea directly: `l=0` is flat, while higher `l` blocks contain richer angular variation. The `2l+1` curves in one row are not separate unrelated features; together they are the components of one rotational block. This is a one-dimensional slice of the sphere, chosen because it makes the angular-frequency intuition visible. The accompanying sphere plot restores the missing latitude direction and shows the same components as colored patterns over the whole surface.

In [46]:
x

tensor([[-0.1297, -0.8638, -0.4869],
        [-0.6960,  0.4131,  0.5873],
        [ 0.3686,  0.2950,  0.8815],
        [-0.6661,  0.1345, -0.7336],
        [ 0.3800,  0.3163,  0.8692],
        [-0.1974,  0.9623,  0.1872],
        [ 0.4368,  0.5132,  0.7388],
        [-0.4635, -0.8757,  0.1354],
        [ 0.8264, -0.2907,  0.4822],
        [ 0.7614,  0.6327, -0.1412]])

In [47]:

def generate_spherical_harmonics(l,
                                 coords,
                                 normalize=True,
                                 normalization='integral'
                                 ):
    return o3.spherical_harmonics(
        l,
        coords,
        normalize,
        normalization
    )

n=3

l = {}
for i in range(n+1):
    l[i] = generate_spherical_harmonics(i, x)

In [48]:
print(l[0])
print(l[1])

tensor([[0.2821],
        [0.2821],
        [0.2821],
        [0.2821],
        [0.2821],
        [0.2821],
        [0.2821],
        [0.2821],
        [0.2821],
        [0.2821]])
tensor([[-0.0634, -0.4221, -0.2379],
        [-0.3401,  0.2018,  0.2869],
        [ 0.1801,  0.1441,  0.4307],
        [-0.3255,  0.0657, -0.3585],
        [ 0.1857,  0.1546,  0.4247],
        [-0.0965,  0.4702,  0.0914],
        [ 0.2134,  0.2507,  0.3610],
        [-0.2264, -0.4279,  0.0662],
        [ 0.4038, -0.1421,  0.2356],
        [ 0.3720,  0.3091, -0.0690]])


### The same $l=2$ information in e3nn's real basis

For the integral-normalized convention used here, e3nn's five real $l=2$ components are proportional to the following quadratic patterns, in the library's output order:

$$
\left(
\sqrt{\frac{15}{4\pi}}xz,
\sqrt{\frac{15}{4\pi}}xy,
\sqrt{\frac{5}{16\pi}}(2y^2-x^2-z^2),
\sqrt{\frac{15}{4\pi}}yz,
\sqrt{\frac{15}{16\pi}}(z^2-x^2)
\right).
$$

These expressions are not five new physical quantities. They are five particular linear combinations of the entries of $T_{\mathrm{ST}}$. There is therefore no separate ‘e3nn $Q$’. The geometric object is still

$$
Q:=T_{\mathrm{ST}}=\hat r\hat r^{\mathsf T}-\frac{I}{3}.
$$

What changes is only the coordinate system used to record its five independent numbers. Let

$$
q_{\mathrm{cart}}=(Q_{xx},Q_{yy},Q_{xy},Q_{xz},Q_{yz})^{\mathsf T}.
$$

Because $Q$ is traceless, $Q_{zz}=-Q_{xx}-Q_{yy}$. Define

$$
\alpha=\sqrt{\frac{15}{4\pi}},
\qquad
\beta=\sqrt{\frac{5}{16\pi}},
\qquad
\gamma=\sqrt{\frac{15}{16\pi}}.
$$

Then e3nn's real $l=2$ coordinate vector is obtained by one fixed matrix multiplication:

$$
Y^{(2)}_{\mathrm{e3nn}}(\hat r)
=Aq_{\mathrm{cart}},
\qquad
A=
\begin{bmatrix}
0&0&0&\alpha&0\\
0&0&\alpha&0&0\\
0&3\beta&0&0&0\\
0&0&0&0&\alpha\\
-2\gamma&-\gamma&0&0&0
\end{bmatrix}.
$$

For example, the third e3nn component initially looks unrelated to $Q$:

$$
\beta(2y^2-x^2-z^2).
$$

Substitute $x^2=Q_{xx}+1/3$, $y^2=Q_{yy}+1/3$, and $z^2=Q_{zz}+1/3$. The constant terms cancel, leaving

$$
\beta(2Q_{yy}-Q_{xx}-Q_{zz})=3\beta Q_{yy}.
$$

The fifth component similarly becomes $\gamma(Q_{zz}-Q_{xx})=\gamma(-2Q_{xx}-Q_{yy})$. The other three components are scaled versions of $Q_{xz}$, $Q_{xy}$, and $Q_{yz}$. This is the missing Cartesian-to-e3nn bridge.

The normalization makes e3nn's spherical functions orthonormal on the sphere, while the chosen combinations make rotations act through the standard real $l=2$ representation. The matrix $A$ is constant and invertible. It is not an orthogonal matrix under the naive Euclidean metric on our five-number list because $(Q_{xx},Q_{yy},Q_{xy},Q_{xz},Q_{yz})$ was a convenient independent list, not an orthonormal Cartesian basis.

The two descriptions also have equivalent rotation laws:

$$
Q' = RQR^{\mathsf T}
\qquad\Longleftrightarrow\qquad
Y^{(2)\prime}_{\mathrm{e3nn}}=D^{(2)}(R)Y^{(2)}_{\mathrm{e3nn}}.
$$

This is why the sphere plots show lobes: each color pattern is simply the positive and negative regions of one of these quadratic coordinate functions.

> **Takeaway.** e3nn does not create a new $Q$; it expresses the same $l=2$ tensor in a normalized real spherical basis.

For $\hat r=(1/\sqrt2,1/\sqrt2,0)$, the Cartesian five-number list is $(1/6,1/6,1/2,0,0)$. In e3nn's basis, the same direction produces approximately $(0,0.5463,0.1577,0,-0.2731)$. The numbers differ because the coordinate bases differ; both five-component blocks transform as $l=2$.


In [49]:
# CHECK — reproduce e3nn's l=2 output from its documented real-basis polynomials.
x_coord, y_coord, z_coord = r_hat
manual_e3nn_l2 = torch.stack(
    [
        math.sqrt(15 / (4 * math.pi)) * x_coord * z_coord,
        math.sqrt(15 / (4 * math.pi)) * x_coord * y_coord,
        math.sqrt(5 / (16 * math.pi))
        * (2 * y_coord**2 - x_coord**2 - z_coord**2),
        math.sqrt(15 / (4 * math.pi)) * y_coord * z_coord,
        math.sqrt(15 / (16 * math.pi)) * (z_coord**2 - x_coord**2),
    ]
)
library_e3nn_l2 = generate_spherical_harmonics(2, r_hat)

alpha = math.sqrt(15 / (4 * math.pi))
beta = math.sqrt(5 / (16 * math.pi))
gamma = math.sqrt(15 / (16 * math.pi))
cartesian_to_e3nn = torch.tensor(
    [
        [0, 0, 0, alpha, 0],
        [0, 0, alpha, 0, 0],
        [0, 3 * beta, 0, 0, 0],
        [0, 0, 0, 0, alpha],
        [-2 * gamma, -gamma, 0, 0, 0],
    ],
    dtype=r_hat.dtype,
)
mapped_e3nn_l2 = cartesian_to_e3nn @ l2_cartesian

assert torch.allclose(manual_e3nn_l2, library_e3nn_l2, atol=1e-6)
assert torch.allclose(mapped_e3nn_l2, library_e3nn_l2, atol=1e-6)
print("Cartesian five-number list:", l2_cartesian)
print("A @ Cartesian list:", mapped_e3nn_l2)
print("e3nn normalized real basis:", library_e3nn_l2)


Cartesian five-number list: tensor([0.1667, 0.1667, 0.5000, 0.0000, 0.0000])
A @ Cartesian list: tensor([ 0.0000,  0.5463,  0.1577,  0.0000, -0.2731])
e3nn normalized real basis: tensor([ 0.0000,  0.5463,  0.1577,  0.0000, -0.2731])


In [50]:
from plotly.subplots import make_subplots

def plot_harmonics_on_equator(max_l=3, samples=361):
    """Show a Fourier-like azimuthal sweep of each angular block."""
    phi = torch.linspace(0, 2 * math.pi, samples)
    directions = torch.stack(
        (torch.cos(phi), torch.sin(phi), torch.zeros_like(phi)),
        dim=1,
    )
    circle_blocks = {
        l_order: generate_spherical_harmonics(l_order, directions)
        for l_order in range(max_l + 1)
    }

    fig = make_subplots(
        rows=max_l + 1,
        cols=1,
        shared_xaxes=True,
        vertical_spacing=0.04,
        subplot_titles=[
            f"l={l_order}: {2 * l_order + 1} components"
            for l_order in range(max_l + 1)
        ],
    )
    phi_degrees = torch.rad2deg(phi).tolist()
    for l_order, block in circle_blocks.items():
        for component in range(block.shape[1]):
            fig.add_trace(
                go.Scatter(
                    x=phi_degrees,
                    y=block[:, component].detach().cpu().tolist(),
                    mode="lines",
                    name=f"component {component}",
                    legendgroup=f"l={l_order}",
                    showlegend=l_order == 1,
                ),
                row=l_order + 1,
                col=1,
            )
    fig.update_xaxes(title_text="azimuth around the circle (degrees)", row=max_l + 1, col=1)
    fig.update_yaxes(title_text="basis value")
    fig.update_layout(
        title="Spherical harmonics as Fourier-like angular modes",
        height=230 * (max_l + 1),
        width=850,
        hovermode="x unified",
        margin={"l": 60, "r": 20, "b": 60, "t": 70},
    )
    display(HTML(fig.to_html(full_html=False, include_plotlyjs="cdn")))
    return circle_blocks

circle_blocks = plot_harmonics_on_equator(max_l=3)

def plot_harmonics_on_sphere(l_orders=(0, 1, 2), n_theta=31, n_phi=61):
    """Display spherical-harmonic values as colored sphere surfaces."""
    theta = torch.linspace(0, math.pi, n_theta)
    phi = torch.linspace(0, 2 * math.pi, n_phi)
    theta_grid, phi_grid = torch.meshgrid(theta, phi, indexing="ij")
    sphere_xyz = torch.stack(
        (
            torch.sin(theta_grid) * torch.cos(phi_grid),
            torch.sin(theta_grid) * torch.sin(phi_grid),
            torch.cos(theta_grid),
        ),
        dim=-1,
    )
    flat_xyz = sphere_xyz.reshape(-1, 3)
    sphere_blocks = {
        l_order: generate_spherical_harmonics(l_order, flat_xyz).reshape(
            n_theta, n_phi, 2 * l_order + 1
        )
        for l_order in l_orders
    }

    panels = [(l_order, component)
              for l_order in l_orders
              for component in range(2 * l_order + 1)]
    columns = 2
    rows = math.ceil(len(panels) / columns)
    specs = [
        [
            {"type": "surface"} if row * columns + col < len(panels) else None
            for col in range(columns)
        ]
        for row in range(rows)
    ]
    fig = make_subplots(rows=rows, cols=columns, specs=specs,
                        subplot_titles=[f"l={l_order}, component {component}"
                                        for l_order, component in panels])

    for index, (l_order, component) in enumerate(panels):
        row, col = divmod(index, columns)
        values = sphere_blocks[l_order][:, :, component]
        limit = max(float(values.abs().max()), 1e-8)
        fig.add_trace(
            go.Surface(
                x=sphere_xyz[:, :, 0].tolist(),
                y=sphere_xyz[:, :, 1].tolist(),
                z=sphere_xyz[:, :, 2].tolist(),
                surfacecolor=values.tolist(),
                colorscale="RdBu",
                cmin=-limit,
                cmax=limit,
                showscale=index == 0,
                colorbar={"title": "value"},
                hovertemplate="%{surfacecolor:.3f}<extra></extra>",
            ),
            row=row + 1,
            col=col + 1,
        )
    fig.update_scenes(aspectmode="data")
    fig.update_layout(
        title="Spherical harmonics on the full sphere",
        height=360 * rows,
        width=850,
        showlegend=False,
        hoverlabel={"align": "left", "font": {"size": 14}},
        margin={"l": 0, "r": 0, "b": 0, "t": 70},
    )
    display(HTML(fig.to_html(full_html=False, include_plotlyjs="cdn")))
    return sphere_blocks

sphere_blocks = plot_harmonics_on_sphere()

In [51]:
# LAB 8A — graph angular orders
# TODO (you): sample directions on the unit sphere.
# TODO (you): evaluate a documented real spherical-harmonic basis for l = 0, 1, 2.
# TODO (you): plot every m component with clear labels and a signed-value legend.
# TODO (you): for l = 1, compare the basis values with normalized x, y, z.
# CHECK: the number of plots for each l equals 2l + 1.

In [52]:
# LAB 8B — basis checks
# TODO (you): estimate the Gram matrix of the sampled basis functions.
# TODO (you): compare it with the result expected from your normalization convention.
# CHECK: different basis functions are approximately orthogonal.

## 9. Wigner-D matrices

### Your explanation

Use the plots above to motivate Wigner-D. Explain what rotates physically, what rotates in feature space, why `Dˡ(R)` has shape `(2l + 1) × (2l + 1)`, and what the identity `Yˡ(Rr̂) = Dˡ(R)Yˡ(r̂)` claims.

### Blog paragraph

In [53]:
R

tensor([[ 0.7071, -0.7071,  0.0000],
        [ 0.7071,  0.7071,  0.0000],
        [ 0.0000,  0.0000,  1.0000]])

In [54]:
def generate_wigner_D(l,R):
    return o3.Irrep(l, 1).D_from_matrix(R)

D = {}
for i in range(3):
    D[i] = generate_wigner_D(i, R)

print([D[i].shape for i in range(3)])



[torch.Size([1, 1]), torch.Size([3, 3]), torch.Size([5, 5])]


In [55]:
# LAB 9A — inspect Wigner-D actions
# TODO (you): construct D⁰, D¹, and D² using the same basis conventions as Lab 8.
# TODO (you): print shapes and visualize how one feature block changes under rotation.
# CHECK: every Dˡ has shape (2l + 1, 2l + 1).

In [56]:
Y2_r_hat = generate_spherical_harmonics(2, r_hat)
r_hat_rot = R @ r_hat
Y2_r_hat_rot = generate_spherical_harmonics(2, r_hat_rot)
print(r_hat, Y2_r_hat)
print(r_hat_rot, Y2_r_hat_rot)
Y2_pred = D[2] @ Y2_r_hat
print(Y2_pred)
torch.max(torch.abs(Y2_pred - Y2_r_hat_rot))

tensor([0.7071, 0.7071, 0.0000]) tensor([ 0.0000,  0.5463,  0.1577,  0.0000, -0.2731])
tensor([0.0000, 1.0000, 0.0000]) tensor([0.0000, 0.0000, 0.6308, 0.0000, 0.0000])
tensor([-1.2663e-07,  1.1534e-07,  6.3078e-01,  3.1082e-07,  1.1921e-07])


tensor(3.1082e-07)

In [57]:
# LAB 9B — numerical equivariance
# TODO (you): compare Yˡ(Rr̂) with Dˡ(R)Yˡ(r̂) for random directions.
# TODO (you): test Dˡ(R₁R₂) = Dˡ(R₁)Dˡ(R₂).
# TODO (you): report absolute and relative errors for every l.
# CHECK: both identities hold within your declared tolerance.

## 10. Feature channels versus angular components

### Your explanation

Distinguish channel `k`, angular order `l`, and component `m`. Explain `W[k, k̃, l]` as a learned channel map whose action is shared across every `m` component of one `l` block.

### ML bridge

Connect this to an ordinary dense layer with a symmetry-required tied axis.

### Blog paragraph

### Blog-ready explanation

It is easy to confuse a feature channel with an angular component because both appear as axes of one tensor. They do different jobs. The index $l$ selects the rotation type: $l=0$ is scalar-like, $l=1$ is vector-like, and $l=2$ is the five-component symmetric-traceless type. The index $m$ then identifies one of the $2l+1$ components *inside* that chosen rotation type. A channel index, written $k$, is different again: it labels separate learned copies of the same kind of geometric feature.

For example, suppose we hold $l=2$ fixed. Two input channels then form a feature array $H_{\tilde k m}$ of shape $(2,5)$: two learned copies of a five-component $l=2$ object. A learned channel map $W_{k\tilde k}$ of shape $(3,2)$ creates three output channels:

$$
H'_{km}=\sum_{\tilde k}W_{k\tilde k}H_{\tilde k m}.
$$

The result has shape $(3,5)$. The matrix $W$ decides how to combine the two input channels into three output channels, but it applies exactly the same channel mixture at every $m$. In ordinary deep-learning language, this is a dense layer across channels with weights tied across the angular-component axis.

That tying is not a cosmetic implementation choice. Under rotation, the five $m$ components mix through the same Wigner-$D^{(2)}$ matrix. Because $W$ never singles out a particular $m$, channel mixing commutes with that rotation. If we gave each $m$ its own independent weight matrix, the channel map would generally fail to commute with Wigner-$D$, and the layer would no longer be equivariant.


In [58]:
Y2_r_hat_rot

tensor([0.0000, 0.0000, 0.6308, 0.0000, 0.0000])

In [59]:
mix = torch.nn.Linear(2,3,False)
Hin = torch.concat([Y2_r_hat.reshape(1,-1), Y2_r_hat_rot.reshape(1,-1)], dim=0)
Hin.shape
Hout = mix(Hin.T).T
Hout.shape

print(mix(D[2] @ Hin.T))
# print(D[2] @ mix(Hin.T).T)


tensor([[-4.4540e-07, -7.8161e-07, -7.4584e-07],
        [ 2.3226e-01,  3.8152e-01,  3.2813e-01],
        [-1.9299e-01, -6.8036e-02,  3.0809e-01],
        [ 2.3469e-10,  1.2307e-07,  2.8650e-07],
        [ 1.1613e-01,  1.9076e-01,  1.6406e-01]], grad_fn=<MmBackward0>)


In [60]:
# LAB 10 — channel mixing: valid shared weights versus invalid m-dependent weights.
D2 = D[2].to(dtype=Hin.dtype)

# Each row is an l=2 feature block, so rotation acts on the right.
Hin_rotated = Hin @ D2.T
Hout_from_rotated_input = mix(Hin_rotated.T).T
Hout_then_rotated = Hout @ D2.T
shared_weight_error = torch.max(
    torch.abs(Hout_from_rotated_input - Hout_then_rotated)
)

# Give every m component its own channel map: this should break equivariance.
torch.manual_seed(0)
m_dependent_weights = torch.randn(5, 3, 2, dtype=Hin.dtype)

def m_dependent_mix(features):
    return torch.einsum("moi,im->om", m_dependent_weights, features)

Hout_bad = m_dependent_mix(Hin)
Hout_bad_from_rotated_input = m_dependent_mix(Hin_rotated)
Hout_bad_then_rotated = Hout_bad @ D2.T
m_dependent_error = torch.max(
    torch.abs(Hout_bad_from_rotated_input - Hout_bad_then_rotated)
)

assert torch.allclose(
    Hout_from_rotated_input, Hout_then_rotated, atol=1e-5
)
assert m_dependent_error > 1e-4

print(f"{'channel map':<22} | max equivariance error")
print(f"{'shared across m':<22} | {shared_weight_error:.2e}")
print(f"{'m-dependent':<22} | {m_dependent_error:.2e}")

channel map            | max equivariance error
shared across m        | 2.98e-08
m-dependent            | 1.86e+00


## 11. Why products create new rotation types

### The question that remains

So far, we have learned two complementary lessons. Spherical harmonics manufacture features of a chosen rotation type $l$, and channel mixing can combine learned channels *within* a fixed $l$ block. That is deliberately restrictive: an equivariant linear map cannot freely turn an $l=1$ vector-like feature into an $l=2$ feature. To create richer geometric information, we need a new operation: multiply two equivariant features.

Take the simplest case: two $l=1$ vectors, $v$ and $w$. Their pairwise products form the outer product

$$
T=v\otimes w,\qquad T_{ij}=v_iw_j.
$$

This gives nine values. We already proved in Section 4 that they transform equivariantly as $T\mapsto RTR^{\mathsf T}$. The new point is how to *read* that same fact: $T$ is one feature living in the nine-dimensional tensor-product representation

$$
l=1\otimes l=1.
$$

In flattened coordinates, a rotation acts by $D^{(1)}(R)\otimes D^{(1)}(R)$ on those nine values. This is the representation-theory version of the familiar Cartesian rule $RTR^{\mathsf T}$.

### Why the product is not finished yet

The nine-dimensional product is equivariant, but it is not irreducible: it contains several distinct rotation types mixed together. This is not a new defect in the outer product. It is exactly the structure we discovered earlier when we split a $3\times3$ tensor into trace, antisymmetric, and symmetric-traceless pieces.

Section 12 reinterprets that old decomposition as the answer to a new question: given two input irreps, which irreps can their product contain, and how do we extract each one? Clebsch–Gordan coefficients are the general rule for performing that extraction.

### Blog-ready takeaway

Linear channel mixing preserves a feature's rotation type. Tensor products are the mechanism that create richer types: multiplying two $l=1$ vector blocks gives a nine-component equivariant product space. That space is still reducible, so the next step is to reorganise it into its independent $l=0$, $l=1$, and $l=2$ parts. This is the job of Clebsch–Gordan decomposition.

### Checkpoint — no new code

Section 4 already established the only numerical fact needed here:

$$
(Rv)\otimes(Rw)=R(v\otimes w)R^{\mathsf T}.
$$

Keep that code as the Cartesian proof. The sole new annotation is: its input is $l=1\otimes l=1$, and its nine-value output is an equivariant but reducible product representation. The actual decomposition begins next.

## 12. Clebsch–Gordan decomposition

### Start from the decomposition we already know

Section 4 already decomposed the Cartesian outer product $T=v\otimes w$ into trace, antisymmetric, and symmetric-traceless pieces. Do **not** redo that work here. This section gives that same decomposition its representation-theory meaning: it is the simplest Clebsch–Gordan (CG) change of basis.

For two ordinary vectors, the decomposition is:

| Piece of $T=v\otimes w$ | Output rotation type | Independent components |
| --- | --- | ---: |
| Trace part $T_{\mathrm{tr}}$ | $l=0$ | 1 |
| Antisymmetric part $T_{\mathrm{anti}}$ | $l=1$ | 3 |
| Symmetric-traceless part $T_{\mathrm{ST}}$ | $l=2$ | 5 |

$$
1\otimes1=0\oplus1\oplus2,
\qquad
9=1+3+5.
$$

No new tensor is being created here: Clebsch–Gordan decomposition simply reorganises the same nine outer-product values into three blocks that rotate independently.

### Blog paragraph

#### From a pile of products to meaningful geometric features

An outer product is the most literal way to let two geometric features interact. If $v$ and $w$ are vector features, their product $T=v\otimes w$ contains every pairwise coordinate product: $v_xw_x$, $v_xw_y$, and so on. There are $3\times3=9$ such values. The product transforms correctly when we rotate the molecule, but its nine numbers are initially just a mixed pile of different rotation behaviours.

The useful move is to reorganise that pile into blocks that do not mix under rotation. For two vectors, the trace of $T$ is one scalar number, the antisymmetric part behaves like a three-component vector, and the symmetric-traceless part is a five-component quadrupole. In the language of irreducible representations, this is

$$
1\otimes1=0\oplus1\oplus2.
$$

The dimensions provide a reassuring accounting check: the original $9$ values have not disappeared or been approximated; they have been regrouped as $9=1+3+5$. This is why a tensor product can make a network more expressive without breaking rotational equivariance: it creates richer geometric information, then keeps each distinct rotation type in its own block.

This pattern is completely general. Combining an $l_1$ block with an $l_2$ block can produce every total angular order from $|l_1-l_2|$ through $l_1+l_2$:

$$
l_1\otimes l_2=\bigoplus_{L=|l_1-l_2|}^{l_1+l_2}L.
$$

For example, a vector block has three components and an $l=2$ block has five, so their $15$ raw pairwise products reorganise as

$$
1\otimes2=1\oplus2\oplus3,
\qquad
3\times5=3+5+7.
$$

Clebsch–Gordan coefficients are the fixed, symmetry-dictated recipe for this regrouping. They take pairs of input components labelled $(m_1,m_2)$ and form output components labelled $(L,M)$. They are not learned parameters: geometry determines them once we choose a basis convention. A MACE layer learns which *channels* to combine; Clebsch–Gordan coefficients determine which combinations have a clean, well-defined rotation type.

The component label is conserved: $M=m_1+m_2$. But $M$ does not uniquely determine the output rotation type. For example, $M=1$ may live in an $L=1$, $L=2$, or $L=3$ block; the Clebsch–Gordan coefficients separate those distinct rotation behaviours.

> **Curious reader?** The standard derivation treats CG coefficients as the change of basis between the raw product basis and total-angular-order basis. [These notes on angular-momentum addition](https://etneil.github.io/grad_qm_lec_notes/addition_J.html) give the full route; it is optional for following MACE.

The lab below makes this fixed change of basis concrete for $1\otimes1$. Later, when we use the complex spherical basis, the same idea is written with the component labels $(m_1,m_2)$ and $(L,M)$; there the exact selection rule is $M=m_1+m_2$.

In [61]:
print(torch.trace(torch.outer(v,w)))

tensor(105.)


The trace is the unnormalised $l=0$ coordinate: rotating both input vectors leaves $\operatorname{tr}(v\otimes w)=v\cdot w$ unchanged. The normalised scalar used in the lab is $s=\operatorname{tr}(T)/\sqrt3$.

### Lab convention: one explicit real CG-style matrix

This lab uses a real Cartesian basis, not e3nn's real spherical-harmonic basis. That is deliberate: the geometry is visible here. Different valid basis conventions may change signs or rotate coordinates *within* one $l$-block, but never mix the $l=0$, $l=1$, and $l=2$ blocks.

Flatten the raw product row-by-row:

$$
t=(T_{xx},T_{xy},T_{xz},T_{yx},T_{yy},T_{yz},T_{zx},T_{zy},T_{zz})^\mathsf{T}.
$$

We will create a normalized output vector $y=C_{\mathrm{cart}}t$ in this order:

$$
y=(s,a_x,a_y,a_z,q_1,q_2,q_3,q_4,q_5)^\mathsf{T},
$$

where

$$
\begin{aligned}
s &= \frac{T_{xx}+T_{yy}+T_{zz}}{\sqrt3},\\
(a_x,a_y,a_z) &= \frac{1}{\sqrt2}(T_{yz}-T_{zy},\;T_{zx}-T_{xz},\;T_{xy}-T_{yx}),\\
(q_1,q_2,q_3,q_4,q_5) &= \left(\frac{T_{xx}-T_{yy}}{\sqrt2},\;\frac{2T_{zz}-T_{xx}-T_{yy}}{\sqrt6},\;\frac{T_{xy}+T_{yx}}{\sqrt2},\;\frac{T_{xz}+T_{zx}}{\sqrt2},\;\frac{T_{yz}+T_{zy}}{\sqrt2}\right).
\end{aligned}
$$

The first coordinate is the normalized $l=0$ scalar; the next three are the $l=1$ block; the final five are the $l=2$ block. These equations are the rows of $C_{\mathrm{cart}}$. The $1/\sqrt3$ normalization is why reconstructing the trace *matrix* still gives $T_{\mathrm{tr}}=\operatorname{tr}(T)I/3$.

In [62]:
# LAB 12A — CG as a real change of basis for 1 ⊗ 1
# You already decomposed T in Section 4. Do not repeat that calculation.
#
# 1. Form T = torch.outer(v, w), then flatten row-by-row:
#    t = (Txx, Txy, Txz, Tyx, Tyy, Tyz, Tzx, Tzy, Tzz).
# 2. Build the fixed 9×9 matrix C_cart from the convention in the Markdown cell above.
# 3. Compute y = C_cart @ t. Its slices are l=0 (one value), l=1 (three), l=2 (five).
#
# CHECK: C_cart.shape == (9, 9) and y.shape == (9,).
# CHECK: C_cart.T @ C_cart is the 9×9 identity.
# CHECK: C_cart.T @ y reconstructs t.

In [ ]:
T = torch.outer(v,w)
t = T.flatten()
sqrt_3_inv = torch.tensor(3**(-0.5))
c0 = torch.tensor([sqrt_3_inv, 0, 0, 0, sqrt_3_inv, 0, 0, 0, sqrt_3_inv])
c0

s = c0 @ t
print(s)
print(torch.trace(T) / math.sqrt(3))


tensor(60.6218)
tensor(60.6218)
tensor(0.5774)
tensor([0.5774, 0.0000, 0.0000, 0.0000, 0.5774, 0.0000, 0.0000, 0.0000, 0.5774])
tensor(60.6218)


In [81]:
sqrt_2_inv = 2**(-0.5)
cax = torch.tensor([0, 0, 0, 0, 0, sqrt_2_inv, 0, -sqrt_2_inv, 0])
cax
cay = torch.tensor([0, 0, -sqrt_2_inv, 0, 0, 0, sqrt_2_inv, 0, 0])
caz = torch.tensor([0, sqrt_2_inv,0,-sqrt_2_inv, 0, 0, 0, 0, 0])

print(cax.shape, cay.shape, caz.shape)
cl1 = torch.concat([cax.reshape(1,-1), cay.reshape(1,-1), caz.reshape(1,-1)], dim=0)
print(cl1.shape)
print(cl1 @ cl1.T)

torch.Size([9]) torch.Size([9]) torch.Size([9])
torch.Size([3, 9])
tensor([[1.0000, 1.0000, 0.0000],
        [1.0000, 1.0000, 0.0000],
        [0.0000, 0.0000, 1.0000]])


In [83]:
sqrt_6_inv = sqrt_3_inv * sqrt_2_inv
cq1 = torch.tensor([sqrt_2_inv, 0, 0, 0, -sqrt_2_inv, 0, 0, 0, 0])
cq2 = torch.tensor([-sqrt_6_inv, 0, 0, 0, -sqrt_6_inv, 0, 0, 0, 2 * sqrt_6_inv])
cq3 = torch.tensor([0, sqrt_2_inv, 0, sqrt_2_inv, 0, 0, 0, 0, 0])
cq4 = torch.tensor([0, 0, sqrt_2_inv, 0, 0, 0, sqrt_2_inv, 0, 0])
cq5 = torch.tensor([0, 0, 0, 0, 0, sqrt_2_inv, 0, sqrt_2_inv, 0])

C_cart = torch.stack([c0, cax, cay, caz, cq1, cq2, cq3, cq4, cq5])
y = C_cart @ t

assert torch.allclose(C_cart.T @ C_cart, torch.eye(9))
assert torch.allclose(C_cart.T @ y, t)
print(C_cart.shape, y.shape)
print("C_cart.T @ C_cart = I and C_cart.T @ y = t ✓")

tensor([-0.4082,  0.0000,  0.0000, -0.4082,  0.0000,  0.0000,  0.5774,  0.0000,
         0.0000])

In [ ]:
# LAB 12B — read the three output blocks
l0_block, l1_block, l2_block = y[:1], y[1:4], y[4:9]

assert torch.allclose(l0_block[0], torch.trace(T) / math.sqrt(3))
print("l=0 block:", l0_block.shape)
print("l=1 block:", l1_block.shape)
print("l=2 block:", l2_block.shape)
print("The same nine raw products are now organised as 1 + 3 + 5.")

# C_cart is fixed by the rotation symmetry and our chosen normalized basis; it is not learned.
# STOP HERE: complex-basis coefficient tables and the M = m₁ + m₂ selection rule are a later optional deep dive.

## 13. Higher-order coupling: `ν` and `η_ν`

### Your explanation

Using three factors, explain what `ν` counts, how tensor-product order relates to physical body order, why intermediate angular orders appear, and why `η_ν` is needed to distinguish multiple valid coupling paths to the same final `L`. Draw at least two coupling trees.

### Blog paragraph

In [ ]:
# LAB 13 — coupling paths
# TODO (you): enumerate valid intermediate orders for one small ν = 3 example.
# TODO (you): assign η₃ labels and record intermediate/final shapes.
# CHECK: every path obeys the angular-momentum selection rules.

## 14. Radial information

### Your explanation

Explain distance versus direction, fixed Bessel basis functions, the smooth cutoff, learned combinations of fixed basis values, and why radial outputs are rotationally invariant.

### ML bridge

Compare fixed radial expansion followed by learned mixing with a fixed feature map feeding an MLP.

### Blog paragraph

In [ ]:
# LAB 14 — radial basis and cutoff
# TODO (you): graph several Bessel-style basis functions over the cutoff range.
# TODO (you): graph the cutoff and the enveloped basis functions.
# TODO (you): graph one learned linear combination of fixed basis values.
# CHECK: every enveloped basis function smoothly reaches zero at the cutoff.

# Part II — assemble MACE

The remaining sections should feel like composition. Before each equation, list tensors and shapes, name their symmetry type, expand the operations into loops, and connect every operation to a completed lab.

## 15. Eq. 8 — construct `A`

Paste Eq. 8 here. Then write:

- your tensor and shape ledger;
- your inside-out loop expansion;
- your mapping from each operation to a fundamentals lab;
- your explanation for a non-GDL ML practitioner;
- the expected output transformation law.

Your explanation:

In [ ]:
# MACE LAB A — Eq. 8 toy construction
# TODO (you): implement the smallest nontrivial A-feature calculation.
# TODO (you): expose radial, angular, channel-mixed, and coupled intermediates.
# TODO (you): pool over a toy neighborhood.
# CHECK: output shape and equivariance match your written contracts.

## 16. Eq. 9 — first-layer message

Copy Eq. 8 and Eq. 9 above this cell, then derive Eq. 9 line by line.

Your derivation:

- What is special about the first-layer node feature?
- Which indices disappear, and why?
- What happens to the Clebsch–Gordan coefficient?
- What does the remaining neighbor sum compute?

Blog paragraph:

In [ ]:
# MACE LAB B — Eq. 9 as a special case of Eq. 8
# TODO (you): evaluate your Eq. 8 toy calculation with a scalar element feature.
# TODO (you): evaluate the directly simplified Eq. 9 expression on the same inputs.
# CHECK: both routes agree within tolerance.

### Paper-notation checkpoint — generalized CG indices

Map the concrete CG lab above onto the paper's generalized coefficient notation.

Identify the input tuple, output `(L, M)`, and coupling-path label `η_ν`. Record which quantities are indices, fixed coefficients, and learned parameters.

Your explanation:

Your paper-notation translation:

In [ ]:
# PAPER CHECKPOINT — generalized CG notation
# TODO (you): annotate one small generalized coefficient tensor with axis meanings.
# TODO (you): connect one η_ν path to the coupling tree drawn in Lab 11.
# CHECK: every axis and convention is documented before Eq. 10 is implemented.

## 17. Eq. 10 — read it from the inside out

Paste Eq. 10 here. Then write the equivalent loop structure in your own words.

Your explanation of the innermost channel sum:

Your explanation of the product over `ξ`:

Your explanation of the outer angular sum:

Your ν = 2 expansion:

Your blog paragraph:

In [ ]:
# MACE LAB C — Eq. 10 with ν = 2
# TODO (you): mix channels independently for both A factors.
# TODO (you): form the raw product and couple it into selected (L, M) outputs.
# TODO (you): keep the η_ν path axis explicit.
# CHECK: all shapes match the ledger and each B block transforms with Dᴸ.

## 18. Eq. 11 — form the message

Paste Eq. 11 here. Explain which `B` features are combined, which indices disappear, which survive, why the receiving atom's element can affect the weights, and why combining paths with the same output `L` preserves equivariance. Connect the result back to Eq. 7.

Your loop expansion:

Your shape ledger:

Your explanation for a non-GDL ML practitioner:

In [ ]:
# MACE LAB D — Eq. 11 toy message
# TODO (you): combine toy B features across ν and η_ν with learned coefficients.
# TODO (you): preserve the requested output (L, M) block.
# CHECK: the message has the declared shape and transformation law.

## Minimal MACE implementation ladder

Begin only after the equation labs are clear:

- [ ] atomic types and positions
- [ ] cutoff neighbor graph
- [ ] relative distance and direction
- [ ] radial basis and cutoff
- [ ] spherical-harmonic edge features
- [ ] first-layer `A` features
- [ ] `ν = 2` product basis
- [ ] message linear combination
- [ ] node update and residual connection
- [ ] invariant scalar readout
- [ ] total energy as an atom-wise sum
- [ ] forces as the negative position gradient

Before implementing any item, write its input shape, output shape, symmetry contract, and smallest decisive test.

## 19. Global MACE shape ledger — write this yourself

Record the shape and meaning of every axis for:

- `A[i, k, l, m]`
- `w[k, k̃, l]`
- `C[η, L, M, (l₁,m₁), ...]`
- `B[i, η, k, L, M]`

Your shape ledger:


In [ ]:
# GLOBAL SHAPE CHECKS
# TODO (you): assert every shape documented above.
# TODO (you): use deliberately different atom, edge, channel, and component counts
# so that an axis swap cannot pass accidentally.

## 20. Final symmetry checks

Before coding, write the two routes you will compare:

- transform input → run operation
- run operation → transform output

Your explanation of what should match:

Your expected failure modes:

In [ ]:
# FINAL VERIFICATION MATRIX
# TODO (you): translation invariance of relative geometry and scalar energy.
# TODO (you): Wigner-D rotation equivariance of every l block.
# TODO (you): inversion/parity under the declared convention.
# TODO (you): atom-permutation equivariance and total-energy invariance.
# TODO (you): position gradients with the same shape as positions.
# TODO (you): report dtype, absolute error, relative error, and tolerance.

# Blog assembly

Turn the completed notebook into a coherent article for an ML practitioner who is new to GDL.

## Reader promise

What should the reader understand by the end?

## Narrative arc

1. Why ordinary scalar features are insufficient for 3D geometry.
2. How vectors build a nine-component Cartesian tensor.
3. How `9 = 1 + 3 + 5` reveals independent rotational blocks.
4. How representations, irreps, `l`, and `m` name those observations.
5. How spherical harmonics encode direction as Fourier-like modes on the sphere.
6. How Wigner-D matrices express equivariance inside each block.
7. How tensor products create interactions and CG coefficients reorganize them.
8. How MACE assembles the pieces efficiently.

## Editing checklist

- [ ] Every symbol is defined before use.
- [ ] Every abstract term has one concrete example.
- [ ] Every important tensor has a shape.
- [ ] Every transformation claim has a numerical check.
- [ ] Every figure says what the reader should notice.
- [ ] Basis and normalization conventions are explicit.
- [ ] Paper notation appears only after the building block is familiar.

## Your draft

# Pass 3 session log

Append one dated entry after each meaningful session.

## YYYY-MM-DD — topic

**Objective and scope:**

**Reasoning / derivation:**

**Exact cells or artifacts completed:**

**Verification and tolerances:**

**Gotchas / unresolved questions:**

**Next action:**